In [1]:
# ============================================
# PHASE 3: STATISTICAL ANALYSIS
# ============================================

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, norm
import matplotlib.pyplot as plt
import seaborn as sns

# Load cleaned data
df = pd.read_csv('../data/processed/user_level_ab_data_fixed.csv')

print(f"Data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nGroup Distribution:")
print(df['experiment_group'].value_counts())
print(f"\nOverall conversion rate: {df['converted'].mean():.2%}")

Data loaded: 100,000 rows, 8 columns

Group Distribution:
experiment_group
control      60015
variant_a    20067
variant_b    19918
Name: count, dtype: int64

Overall conversion rate: 64.03%


In [2]:
# ============================================
# STATISTICAL FUNCTIONS
# ============================================

def calculate_conversion_pvalue(control_conv, treatment_conv, control_n, treatment_n):
    """
    Calculate p-value for conversion rate comparison
    """
    from statsmodels.stats.proportion import proportions_ztest
    
    count = [int(control_conv * control_n), int(treatment_conv * treatment_n)]
    nobs = [control_n, treatment_n]
    
    z_score, p_value = proportions_ztest(count, nobs, alternative='two-sided')
    return p_value

def calculate_segment_pvalue(df, segment_col, segment_value, variant='variant_a'):
    """
    Calculate p-value for a specific segment using chi-square test
    """
    control_subset = df[(df['experiment_group']=='control') & (df[segment_col]==segment_value)]
    variant_subset = df[(df['experiment_group']==variant) & (df[segment_col]==segment_value)]
    
    control_converted = control_subset['converted'].sum()
    control_not_converted = len(control_subset) - control_converted
    variant_converted = variant_subset['converted'].sum()
    variant_not_converted = len(variant_subset) - variant_converted
    
    contingency = [[control_converted, control_not_converted],
                   [variant_converted, variant_not_converted]]
    
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    return p_value

def calculate_lift(df, variant='variant_a'):
    """
    Calculate lift percentage for a variant compared to control
    """
    control_conv = df[df['experiment_group']=='control']['converted'].mean()
    variant_conv = df[df['experiment_group']==variant]['converted'].mean()
    lift = (variant_conv / control_conv - 1) * 100
    return lift, control_conv, variant_conv

print("Functions loaded successfully")

Functions loaded successfully


In [8]:
# ============================================
# OVERALL A/B TEST RESULTS
# ============================================

print("=" * 60)
print("OVERALL A/B TEST RESULTS")
print("=" * 60)

# Calculate for Variant A
lift_a, control_conv, variant_a_conv = calculate_lift(df, 'variant_a')
p_value_a = calculate_conversion_pvalue(control_conv, variant_a_conv, 
                                         len(df[df['experiment_group']=='control']),
                                         len(df[df['experiment_group']=='variant_a']))

print(f"\nVARIANT A vs CONTROL:")
print(f"  Control conversion: {control_conv:.2%}")
print(f"  Variant A conversion: {variant_a_conv:.2%}")
print(f"  Lift: {lift_a:.2f}%")
print(f"  P-value: {p_value_a:.4f}")
print(f"  Significant at 95%: {'YES' if p_value_a < 0.05 else 'NO'}")

# Calculate for Variant B
lift_b, control_conv, variant_b_conv = calculate_lift(df, 'variant_b')
p_value_b = calculate_conversion_pvalue(control_conv, variant_b_conv,
                                         len(df[df['experiment_group']=='control']),
                                         len(df[df['experiment_group']=='variant_b']))

print(f"\nVARIANT B vs CONTROL:")
print(f"  Control conversion: {control_conv:.2%}")
print(f"  Variant B conversion: {variant_b_conv:.2%}")
print(f"  Lift: {lift_b:.2f}%")
print(f"  P-value: {p_value_b:.4f}")
print(f"  Significant at 95%: {'YES' if p_value_b < 0.05 else 'NO'}")

OVERALL A/B TEST RESULTS

VARIANT A vs CONTROL:
  Control conversion: 63.95%
  Variant A conversion: 64.24%
  Lift: 0.45%
  P-value: 0.4601
  Significant at 95%: NO

VARIANT B vs CONTROL:
  Control conversion: 63.95%
  Variant B conversion: 64.08%
  Lift: 0.21%
  P-value: 0.7366
  Significant at 95%: NO


In [4]:
# ============================================
# SEGMENT STATISTICAL SIGNIFICANCE
# ============================================

print("=" * 60)
print("SEGMENT STATISTICAL SIGNIFICANCE")
print("=" * 60)

segments_to_test = [
    ('device_type', 'mobile', 'variant_a', 'Mobile + Variant A'),
    ('device_type', 'desktop', 'variant_a', 'Desktop + Variant A'),
    ('loyalty_tier', 'Bronze', 'variant_a', 'Bronze + Variant A'),
    ('loyalty_tier', 'Platinum', 'variant_b', 'Platinum + Variant B'),
    ('loyalty_tier', 'Silver', 'variant_a', 'Silver + Variant A'),
]

segment_results = []

for segment_col, segment_value, variant, label in segments_to_test:
    p_val = calculate_segment_pvalue(df, segment_col, segment_value, variant)
    
    # Calculate lift for this segment
    control_subset = df[(df['experiment_group']=='control') & (df[segment_col]==segment_value)]
    variant_subset = df[(df['experiment_group']==variant) & (df[segment_col]==segment_value)]
    
    control_conv = control_subset['converted'].mean()
    variant_conv = variant_subset['converted'].mean()
    lift = (variant_conv / control_conv - 1) * 100 if control_conv > 0 else 0
    
    segment_results.append({
        'Segment': label,
        'Lift (%)': lift,
        'P-Value': p_val,
        'Significant (p<0.05)': p_val < 0.05,
        'Control Conv': f"{control_conv:.2%}",
        'Treatment Conv': f"{variant_conv:.2%}"
    })

segment_results_df = pd.DataFrame(segment_results)
print(segment_results_df.to_string(index=False))

SEGMENT STATISTICAL SIGNIFICANCE
             Segment  Lift (%)  P-Value  Significant (p<0.05) Control Conv Treatment Conv
  Mobile + Variant A  0.659177 0.321886                 False       64.32%         64.74%
 Desktop + Variant A -0.699016 0.685677                 False       61.89%         61.45%
  Bronze + Variant A  0.765207 0.377911                 False       60.00%         60.46%
Platinum + Variant B  1.216480 0.700641                 False       73.70%         74.60%
  Silver + Variant A -0.181247 0.886007                 False       67.76%         67.64%


In [9]:
# ============================================
# MULTIPLE TESTING CORRECTION (BONFERRONI)
# ============================================

from statsmodels.stats.multitest import multipletests

# Extract p-values from segment results
p_values = segment_results_df['P-Value'].values

# Apply Bonferroni correction
rejected, p_corrected, alpha_sidak, alpha_bonferroni = multipletests(
    p_values, 
    alpha=0.05, 
    method='bonferroni'
)

print("=" * 60)
print("BONFERRONI CORRECTION RESULTS")
print("=" * 60)
print(f"Number of tests: {len(p_values)}")
print(f"Original alpha: 0.05")
print(f"Bonferroni corrected alpha: {alpha_bonferroni:.6f}")
print("\nCorrected Results:")

for i, (label, p_orig, p_corr, rejected_flag) in enumerate(zip(
    segment_results_df['Segment'], p_values, p_corrected, rejected
)):
    status = "✅ SIGNIFICANT" if rejected_flag else "❌ NOT SIGNIFICANT"
    print(f"\n{label}:")
    print(f"  Original p-value: {p_orig:.4f}")
    print(f"  Bonferroni p-value: {p_corr:.4f}")
    print(f"  Status: {status}")

# Create updated results with correction
segment_results_df['Bonferroni Significant'] = rejected
segment_results_df['Corrected P-Value'] = p_corrected

BONFERRONI CORRECTION RESULTS
Number of tests: 5
Original alpha: 0.05
Bonferroni corrected alpha: 0.010000

Corrected Results:

Mobile + Variant A:
  Original p-value: 0.3219
  Bonferroni p-value: 1.0000
  Status: ❌ NOT SIGNIFICANT

Desktop + Variant A:
  Original p-value: 0.6857
  Bonferroni p-value: 1.0000
  Status: ❌ NOT SIGNIFICANT

Bronze + Variant A:
  Original p-value: 0.3779
  Bonferroni p-value: 1.0000
  Status: ❌ NOT SIGNIFICANT

Platinum + Variant B:
  Original p-value: 0.7006
  Bonferroni p-value: 1.0000
  Status: ❌ NOT SIGNIFICANT

Silver + Variant A:
  Original p-value: 0.8860
  Bonferroni p-value: 1.0000
  Status: ❌ NOT SIGNIFICANT


In [6]:
# ============================================
# REVENUE IMPACT CALCULATION
# ============================================

print("=" * 60)
print("REVENUE IMPACT CALCULATION")
print("=" * 60)

# Calculate average revenue per converted user
avg_revenue_per_purchase = df[df['converted'] == 1]['revenue'].mean()
print(f"\nAverage revenue per purchase: ${avg_revenue_per_purchase:.2f}")

# Function to calculate incremental revenue for a segment
def calculate_incremental_revenue(df, segment_col, segment_value, variant, lift_percentage):
    """
    Calculate incremental revenue if ship to specific segment
    """
    segment_users = len(df[(df[segment_col] == segment_value)])
    control_conv = df[(df['experiment_group']=='control') & (df[segment_col]==segment_value)]['converted'].mean()
    
    # Calculate additional conversions from lift
    additional_conversions = segment_users * (control_conv * (lift_percentage / 100))
    incremental_revenue = additional_conversions * avg_revenue_per_purchase
    
    return incremental_revenue

# Calculate for winning segments
revenue_impacts = []

# Mobile + Variant A
mobile_lift = segment_results_df[segment_results_df['Segment'] == 'Mobile + Variant A']['Lift (%)'].values[0]
mobile_revenue = calculate_incremental_revenue(df, 'device_type', 'mobile', 'variant_a', mobile_lift)
revenue_impacts.append({
    'Segment': 'Mobile Users',
    'Variant': 'Variant A',
    'Lift (%)': mobile_lift,
    'Incremental Revenue (Annual)': f"${mobile_revenue:,.0f}"
})

# Bronze + Variant A
bronze_lift = segment_results_df[segment_results_df['Segment'] == 'Bronze + Variant A']['Lift (%)'].values[0]
bronze_revenue = calculate_incremental_revenue(df, 'loyalty_tier', 'Bronze', 'variant_a', bronze_lift)
revenue_impacts.append({
    'Segment': 'Bronze Tier',
    'Variant': 'Variant A',
    'Lift (%)': bronze_lift,
    'Incremental Revenue (Annual)': f"${bronze_revenue:,.0f}"
})

# Platinum + Variant B
platinum_lift = segment_results_df[segment_results_df['Segment'] == 'Platinum + Variant B']['Lift (%)'].values[0]
platinum_revenue = calculate_incremental_revenue(df, 'loyalty_tier', 'Platinum', 'variant_b', platinum_lift)
revenue_impacts.append({
    'Segment': 'Platinum Tier',
    'Variant': 'Variant B',
    'Lift (%)': platinum_lift,
    'Incremental Revenue (Annual)': f"${platinum_revenue:,.0f}"
})

revenue_df = pd.DataFrame(revenue_impacts)
print("\n" + "=" * 60)
print("INCREMENTAL REVENUE OPPORTUNITIES")
print("=" * 60)
print(revenue_df.to_string(index=False))

# Total opportunity if ship to all winning segments
total_revenue = mobile_revenue + bronze_revenue + platinum_revenue
print(f"\n💰 TOTAL ANNUAL OPPORTUNITY: ${total_revenue:,.0f}")

REVENUE IMPACT CALCULATION

Average revenue per purchase: $2830.18

INCREMENTAL REVENUE OPPORTUNITIES
      Segment   Variant  Lift (%) Incremental Revenue (Annual)
 Mobile Users Variant A  0.659177                   $1,017,563
  Bronze Tier Variant A  0.765207                     $783,287
Platinum Tier Variant B  1.216480                      $76,577

💰 TOTAL ANNUAL OPPORTUNITY: $1,877,428


In [13]:
# ============================================
# POWER ANALYSIS FOR FOLLOW-UP TEST
# ============================================

from statsmodels.stats.power import NormalIndPower
import math

print("=" * 70)
print("POWER ANALYSIS FOR FOLLOW-UP MOBILE TEST")
print("=" * 70)

# Parameters for follow-up test
baseline_conversion = 0.6432  # Mobile control conversion (64.32%)
minimum_detectable_lift = 0.02  # Want to detect 2% absolute lift
alpha = 0.05  # Significance level
power = 0.80  # 80% chance of detecting effect if it exists

# Calculate effect size (Cohen's h)
from statsmodels.stats.proportion import proportion_effectsize
effect_size = proportion_effectsize(
    baseline_conversion, 
    baseline_conversion + minimum_detectable_lift
)

# Calculate required sample size per group
power_analysis = NormalIndPower()
required_n_per_group = power_analysis.solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    alternative='two-sided'
)

required_n_per_group = math.ceil(required_n_per_group)

print(f"\n📊 INPUT PARAMETERS:")
print(f"   Baseline conversion (mobile): {baseline_conversion:.2%}")
print(f"   Minimum detectable lift: {minimum_detectable_lift:.1%} absolute")
print(f"   Significance level (α): {alpha}")
print(f"   Statistical power (1-β): {power:.0%}")

print(f"\n📐 EFFECT SIZE:")
print(f"   Cohen's h: {effect_size:.4f}")

print(f"\n🔢 REQUIRED SAMPLE SIZE:")
print(f"   Users needed per variant: {required_n_per_group:,}")
print(f"   Total users needed (2 variants): {required_n_per_group * 2:,}")

# Compare with current test
current_mobile_users = len(df[df['device_type'] == 'mobile'])
current_per_group = current_mobile_users / 3  # Split across control + 2 variants

print(f"\n📊 COMPARISON WITH CURRENT TEST:")
print(f"   Current mobile users per group: ~{current_per_group:.0f}")
print(f"   Required per group: {required_n_per_group:,}")
print(f"   Shortfall: {required_n_per_group - current_per_group:.0f} more users per group")
print(f"   Current test was underpowered by factor: {required_n_per_group / current_per_group:.1f}x")

print(f"\n⏱️ TEST DURATION ESTIMATE:")
daily_mobile_users = 5000  # Assume 5,000 mobile users per day
days_needed = (required_n_per_group * 2) / daily_mobile_users
print(f"   Assuming {daily_mobile_users:,} mobile users/day")
print(f"   Estimated test duration: {math.ceil(days_needed)} days")

print("\n" + "=" * 70)
print("✅ POWER ANALYSIS COMPLETE")
print("=" * 70)

POWER ANALYSIS FOR FOLLOW-UP MOBILE TEST

📊 INPUT PARAMETERS:
   Baseline conversion (mobile): 64.32%
   Minimum detectable lift: 2.0% absolute
   Significance level (α): 0.05
   Statistical power (1-β): 80%

📐 EFFECT SIZE:
   Cohen's h: -0.0420

🔢 REQUIRED SAMPLE SIZE:
   Users needed per variant: 8,889
   Total users needed (2 variants): 17,778

📊 COMPARISON WITH CURRENT TEST:
   Current mobile users per group: ~28267
   Required per group: 8,889
   Shortfall: -19378 more users per group
   Current test was underpowered by factor: 0.3x

⏱️ TEST DURATION ESTIMATE:
   Assuming 5,000 mobile users/day
   Estimated test duration: 4 days

✅ POWER ANALYSIS COMPLETE


In [10]:
# ============================================
# PRACTICAL SIGNIFICANCE FRAMEWORK
# ============================================

print("=" * 60)
print("PRACTICAL SIGNIFICANCE ASSESSMENT")
print("=" * 60)

# Define business thresholds (typical for e-commerce)
min_meaningful_lift = 0.5  # 0.5% minimum lift to consider shipping
engineering_cost_estimate = 50000  # $50k estimated engineering cost

def evaluate_practical_significance(lift_percentage, p_value, significant_bonferroni, 
                                    incremental_revenue, segment_name):
    """
    Evaluate if a segment result is practically significant
    """
    print(f"\n{segment_name}:")
    print(f"  Lift: {lift_percentage:.2f}%")
    print(f"  Statistically significant (Bonferroni): {significant_bonferroni}")
    
    if not significant_bonferroni:
        return "❌ NOT RECOMMENDED - Not statistically significant after correction"
    
    if lift_percentage < min_meaningful_lift:
        return "⚠️ NOT RECOMMENDED - Lift below minimum meaningful threshold (0.5%)"
    
    if incremental_revenue < engineering_cost_estimate:
        return f"⚠️ CONSIDER WITH CAUTION - Revenue (${incremental_revenue:,.0f}) < Engineering Cost (${engineering_cost_estimate:,.0f})"
    
    return f"✅ RECOMMEND SHIP - Meets statistical, practical, and business criteria"

# Evaluate each segment
print("\n" + "=" * 60)
print("FINAL RECOMMENDATION BY SEGMENT")
print("=" * 60)

# Get corrected significance
mobile_sig = segment_results_df[segment_results_df['Segment'] == 'Mobile + Variant A']['Bonferroni Significant'].values[0]
bronze_sig = segment_results_df[segment_results_df['Segment'] == 'Bronze + Variant A']['Bonferroni Significant'].values[0]
platinum_sig = segment_results_df[segment_results_df['Segment'] == 'Platinum + Variant B']['Bonferroni Significant'].values[0]

evaluate_practical_significance(mobile_lift, 0.02, mobile_sig, mobile_revenue, "Mobile Users + Variant A")
evaluate_practical_significance(bronze_lift, 0.01, bronze_sig, bronze_revenue, "Bronze Tier + Variant A")
evaluate_practical_significance(platinum_lift, 0.03, platinum_sig, platinum_revenue, "Platinum Tier + Variant B")

PRACTICAL SIGNIFICANCE ASSESSMENT

FINAL RECOMMENDATION BY SEGMENT

Mobile Users + Variant A:
  Lift: 0.66%
  Statistically significant (Bonferroni): False

Bronze Tier + Variant A:
  Lift: 0.77%
  Statistically significant (Bonferroni): False

Platinum Tier + Variant B:
  Lift: 1.22%
  Statistically significant (Bonferroni): False


'❌ NOT RECOMMENDED - Not statistically significant after correction'

In [12]:
# ============================================
# SAVE STATISTICAL RESULTS
# ============================================

import os

# Create results directory
os.makedirs('../data/results', exist_ok=True)

# Save segment results
segment_results_df.to_csv('../data/results/segment_statistical_results.csv', index=False)
print("✅ Saved: segment_statistical_results.csv")

# Save revenue impacts
revenue_df.to_csv('../data/results/revenue_impacts.csv', index=False)
print("✅ Saved: revenue_impacts.csv")

# Save summary statistics
summary_stats = {
    'overall_control_conversion': control_conv,
    'overall_variant_a_conversion': variant_a_conv,
    'overall_variant_b_conversion': variant_b_conv,
    'lift_variant_a': lift_a,
    'lift_variant_b': lift_b,
    'p_value_variant_a': p_value_a,
    'p_value_variant_b': p_value_b,
    'mobile_lift': mobile_lift,
    'bronze_lift': bronze_lift,
    'platinum_lift': platinum_lift,
    'total_annual_opportunity': total_revenue
}

summary_df = pd.DataFrame([summary_stats])
summary_df.to_csv('../data/results/summary_statistics.csv', index=False)
print("✅ Saved: summary_statistics.csv")

print("\n✅ All statistical results saved to data/results/")

✅ Saved: segment_statistical_results.csv
✅ Saved: revenue_impacts.csv
✅ Saved: summary_statistics.csv

✅ All statistical results saved to data/results/


In [1]:
# Run this code to get correct sample size values
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Your parameters
baseline = 0.64  # 64% control conversion
power = 0.80     # 80% power
alpha = 0.05     # 5% significance

# Lifts to calculate (as decimals)
lifts = [0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 0.05]

print("Correct Sample Size Calculations:")
print("-" * 50)

for lift in lifts:
    effect_size = proportion_effectsize(baseline, baseline + lift)
    n = NormalIndPower().solve_power(
        effect_size=effect_size,
        power=power,
        alpha=alpha,
        alternative='two-sided'
    )
    print(f"{lift*100:.1f}% lift → {int(n):,} users per variant")

Correct Sample Size Calculations:
--------------------------------------------------
0.5% lift → 144,225 users per variant
1.0% lift → 35,942 users per variant
1.5% lift → 15,922 users per variant
2.0% lift → 8,926 users per variant
2.5% lift → 5,693 users per variant
3.0% lift → 3,939 users per variant
4.0% lift → 2,199 users per variant
5.0% lift → 1,397 users per variant
